In [1]:
import sys
sys.path.append("..")

import ee
ee.Authenticate()
ee.Initialize(project="tree-fire-510209")

import pandas as pd
from src.fires import load_fire, derive_fire_params
from src.grids import pad_bounds

full = pd.read_parquet("../data/processed/all_cells_current.parquet")
all_uids = full["uid"].unique()
print(f"{len(all_uids)} fires currently in the dataset")


247 fires currently in the dataset


In [2]:
def count_images(aoi, start, end, max_cloud_pct=30):
    coll = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", max_cloud_pct))
    )
    return coll.size().getInfo()

rows = []
for uid in all_uids:
    try:
        fire = load_fire(uid)
        lon, lat = fire.geometry.iloc[0].centroid.x, fire.geometry.iloc[0].centroid.y
        row = pd.Series({"uid": uid, "lon": lon, "lat": lat, "start_date": fire["start_date"].iloc[0]})
        params = derive_fire_params(row)
        aoi = ee.Geometry.Rectangle(list(pad_bounds(fire.total_bounds, 0.15)))
        n_pre = count_images(aoi, *params["pre_fire_window"])
        n_post = count_images(aoi, *params["post_fire_window"])
        rows.append({"uid": uid, "n_pre": n_pre, "n_post": n_post, "n_min": min(n_pre, n_post)})
        print(uid, n_pre, n_post)
    except Exception as e:
        print(uid, "FAILED:", e)

image_counts = pd.DataFrame(rows).sort_values("n_min")
image_counts.to_csv("../data/processed/severity_image_counts.csv", index=False)
print(image_counts.head(20))
print(f"\n{(image_counts['n_min'] <= 2).sum()} fires with 2 or fewer images in their sparser window")


2020_112 83 94
2020_135 26 26
2020_464 14 7


/home/jupyter/fire-impact-curves/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: ../data/fire_atlas/SHP_perimeters/GFA_v20240409_perimeters_2021.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


2021_190 147 140
2020_841874 45 55
2020_838262 23 27
2020_818141 84 88


/home/jupyter/fire-impact-curves/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: ../data/fire_atlas/SHP_perimeters/GFA_v20240409_perimeters_2021.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


2021_750951 99 88
2020_325095 36 22
2020_66789 63 45
2020_901450 47 51


/home/jupyter/fire-impact-curves/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: ../data/fire_atlas/SHP_perimeters/GFA_v20240409_perimeters_2021.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


... (143 outputs left out)

/home/jupyter/fire-impact-curves/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: ../data/fire_atlas/SHP_perimeters/GFA_v20240409_perimeters_2021.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


2021_401856 8 10
2020_871140 49 31
2020_858340 24 10
2020_834756 62 52
2020_858341 24 11
2020_830179 12 8
2020_839832 26 24
2020_832636 45 37
2020_833726 25 20
2020_831248 53 52
2020_880556 35 32
2019_873717 17 8


/home/jupyter/fire-impact-curves/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: ../data/fire_atlas/SHP_perimeters/GFA_v20240409_perimeters_2021.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


2021_766525 7 13
2019_873471 19 9
2020_880548 61 34
2019_873714 36 15
2019_873720 15 7
2019_873593 22 9
2020_877292 11 3
2020_880433 38 26
             uid  n_pre  n_post  n_min
13    2021_12806      9       1      1
18    2021_17228      2       6      2
111  2022_760657      3       2      2
107  2020_876837      8       2      2
136  2020_279667      4       2      2
68   2020_233896      7       3      3
179  2021_656640     18       3      3
245  2020_877292     11       3      3
183  2020_700550     11       4      4
80   2020_810300     34       4      4
112  2020_880551      6       4      4
152  2021_771670      7       4      4
81   2021_745780      8       4      4
67   2020_453366     14       5      5
36    2020_14027      5      11      5
108  2020_876693      5       6      5
91   2020_235778      6      11      6
66   2020_273534      9       6      6
142  2020_713150      6       9      6
86   2021_737791     13       7      7

5 fires with 2 or fewer images in their s

In [10]:
import os

thin_fires = ["2021_12806", "2021_17228", "2022_760657", "2020_876837", "2020_279667"]

for uid in thin_fires:
    path = f"../data/severity/severity_{uid}.tif"
    if os.path.exists(path):
        os.remove(path)
        print(f"removed cached {path}")
    else:
        print(f"{path} did not exist")

../data/severity/severity_2021_12806.tif did not exist
../data/severity/severity_2021_17228.tif did not exist
../data/severity/severity_2022_760657.tif did not exist
../data/severity/severity_2020_876837.tif did not exist
../data/severity/severity_2020_279667.tif did not exist


In [9]:
thin_fires = ["2021_12806", "2021_17228", "2022_760657", "2020_876837", "2020_279667"]

for window_days in [75, 150, 365]:
    print(f"--- window_days={window_days} ---")
    for uid in thin_fires:
        fire = load_fire(uid)
        lon, lat = fire.geometry.iloc[0].centroid.x, fire.geometry.iloc[0].centroid.y
        row = pd.Series({"uid": uid, "lon": lon, "lat": lat, "start_date": fire["start_date"].iloc[0]})
        params = derive_fire_params(row, window_days=window_days)
        aoi = ee.Geometry.Rectangle(list(pad_bounds(fire.total_bounds, 0.15)))
        n_pre = count_images(aoi, *params["pre_fire_window"])
        n_post = count_images(aoi, *params["post_fire_window"])
        print(f"  {uid}: pre={n_pre}, post={n_post}")


--- window_days=75 ---


/home/jupyter/fire-impact-curves/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: ../data/fire_atlas/SHP_perimeters/GFA_v20240409_perimeters_2021.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


  2021_12806: pre=9, post=1


/home/jupyter/fire-impact-curves/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: ../data/fire_atlas/SHP_perimeters/GFA_v20240409_perimeters_2021.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


  2021_17228: pre=2, post=6
  2022_760657: pre=3, post=2
  2020_876837: pre=8, post=2
  2020_279667: pre=4, post=2
--- window_days=150 ---


/home/jupyter/fire-impact-curves/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: ../data/fire_atlas/SHP_perimeters/GFA_v20240409_perimeters_2021.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


... (3 outputs left out)

/home/jupyter/fire-impact-curves/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: ../data/fire_atlas/SHP_perimeters/GFA_v20240409_perimeters_2021.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


  2021_12806: pre=34, post=34


/home/jupyter/fire-impact-curves/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: ../data/fire_atlas/SHP_perimeters/GFA_v20240409_perimeters_2021.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


  2021_17228: pre=22, post=23
  2022_760657: pre=33, post=28
  2020_876837: pre=29, post=25
  2020_279667: pre=25, post=20


In [12]:
from src.pipeline import submit_severity_tasks, wait_and_download_severity, build_cell_table
from src.fires import utm_epsg_for

thin_fires = ["2021_12806", "2021_17228", "2022_760657", "2020_876837", "2020_279667"]

thin_configs = []
for uid in thin_fires:
    fire = load_fire(uid)
    lon, lat = fire.geometry.iloc[0].centroid.x, fire.geometry.iloc[0].centroid.y
    row = pd.Series({"uid": uid, "lon": lon, "lat": lat, "start_date": fire["start_date"].iloc[0]})
    thin_configs.append(derive_fire_params(row, window_days=150))

submitted_thin = submit_severity_tasks(thin_configs)
submitted_thin = wait_and_download_severity(submitted_thin)

thin_results = {}
for cfg in thin_configs:
    uid = cfg["uid"]
    lon, lat = cfg.pop("lon"), cfg.pop("lat")
    try:
        thin_results[uid] = build_cell_table(
            utm_epsg=utm_epsg_for(lon, lat),
            severity_path=submitted_thin[uid]["severity_path"],
            **cfg,
        )
        print(uid, "rebuilt:", len(thin_results[uid]), "cells")
    except Exception as e:
        print(uid, "FAILED:", e)
    cfg["lon"], cfg["lat"] = lon, lat

# swap, not append: these uids already exist in full with the old, narrow-window severity
full_v2 = pd.concat(
    [full[~full["uid"].isin(thin_fires)]] + [df.assign(uid=uid) for uid, df in thin_results.items()],
    ignore_index=True,
)
full_v2["height_class"] = pd.cut(full_v2["height"], [0, 10, 20, 100], labels=["<10m", "10-20m", ">20m"])
print(f"full: {len(full)} cells -> full_v2: {len(full_v2)} cells")


/home/jupyter/fire-impact-curves/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: ../data/fire_atlas/SHP_perimeters/GFA_v20240409_perimeters_2021.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(
/home/jupyter/fire-impact-curves/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: ../data/fire_atlas/SHP_perimeters/GFA_v20240409_perimeters_2021.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


reusing cached severity for 2021_12806: ../data/severity/severity_2021_12806.tif
reusing cached severity for 2021_17228: ../data/severity/severity_2021_17228.tif
reusing cached severity for 2022_760657: ../data/severity/severity_2022_760657.tif
reusing cached severity for 2020_876837: ../data/severity/severity_2020_876837.tif
reusing cached severity for 2020_279667: ../data/severity/severity_2020_279667.tif


/home/jupyter/fire-impact-curves/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: ../data/fire_atlas/SHP_perimeters/GFA_v20240409_perimeters_2021.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


reusing cached AGB 2020 for 2021_12806: ../data/cci_agb_ee/agb_2021_12806_2020.tif
reusing cached AGB 2022 for 2021_12806: ../data/cci_agb_ee/agb_2021_12806_2022.tif
reusing cached landcover for 2021_12806: ../data/forest_type/landcover_2021_12806.tif
2021_12806 rebuilt: 459 cells


/home/jupyter/fire-impact-curves/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: ../data/fire_atlas/SHP_perimeters/GFA_v20240409_perimeters_2021.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


reusing cached AGB 2020 for 2021_17228: ../data/cci_agb_ee/agb_2021_17228_2020.tif
reusing cached AGB 2022 for 2021_17228: ../data/cci_agb_ee/agb_2021_17228_2022.tif
reusing cached landcover for 2021_17228: ../data/forest_type/landcover_2021_17228.tif
2021_17228 rebuilt: 467 cells
reusing cached AGB 2020 for 2022_760657: ../data/cci_agb_ee/agb_2022_760657_2020.tif
reusing cached AGB 2022 for 2022_760657: ../data/cci_agb_ee/agb_2022_760657_2022.tif
reusing cached landcover for 2022_760657: ../data/forest_type/landcover_2022_760657.tif
2022_760657 rebuilt: 38 cells
reusing cached AGB 2019 for 2020_876837: ../data/cci_agb_ee/agb_2020_876837_2019.tif
reusing cached AGB 2021 for 2020_876837: ../data/cci_agb_ee/agb_2020_876837_2021.tif
reusing cached AGB 2022 for 2020_876837: ../data/cci_agb_ee/agb_2020_876837_2022.tif
reusing cached landcover for 2020_876837: ../data/forest_type/landcover_2020_876837.tif
2020_876837 rebuilt: 177 cells
reusing cached AGB 2019 for 2020_279667: ../data/cci_agb

In [13]:
print(f"{len(thin_results)} of {len(thin_configs)} thin fires rebuilt successfully")
missing = [uid for uid in thin_fires if uid not in thin_results]
print("missing:", missing)

# confirm these 5 fires' severity actually changed, not just re-ran to the same numbers
for uid in thin_fires:
    if uid in thin_results:
        old_mean = full[full["uid"] == uid]["dnbr"].mean()
        new_mean = full_v2[full_v2["uid"] == uid]["dnbr"].mean()
        print(f"{uid}: old mean dNBR={old_mean:.3f}, new mean dNBR={new_mean:.3f}")


5 of 5 thin fires rebuilt successfully
missing: []
2021_12806: old mean dNBR=0.748, new mean dNBR=0.148
2021_17228: old mean dNBR=0.112, new mean dNBR=0.156
2022_760657: old mean dNBR=0.192, new mean dNBR=0.118
2020_876837: old mean dNBR=0.246, new mean dNBR=0.213
2020_279667: old mean dNBR=0.184, new mean dNBR=0.169


In [14]:
from src.curves import cross_validate_all_groups

affected_groups = set(
    zip(full[full["uid"].isin(thin_fires)]["height_class"],
        full[full["uid"].isin(thin_fires)]["forest_type_label"])
)
print("groups containing these fires:", affected_groups)

cv_before = cross_validate_all_groups(full)
cv_after = cross_validate_all_groups(full_v2)

compare = cv_before.merge(cv_after, on=["height_class", "forest_type"], suffixes=("_before", "_after"))
compare = compare[compare.apply(lambda r: (r["height_class"], r["forest_type"]) in affected_groups, axis=1)]
print(compare[["height_class", "forest_type", "n_folds_before", "mae_before", "bias_before",
                "n_folds_after", "mae_after", "bias_after"]].to_string(index=False))


groups containing these fires: {('>20m', 'deciduous_broadleaf'), ('<10m', 'mixed'), ('<10m', 'deciduous_broadleaf'), ('10-20m', 'deciduous_broadleaf'), (nan, 'unknown'), ('10-20m', 'evergreen_broadleaf'), ('<10m', 'not_forest'), ('>20m', 'evergreen_needleleaf'), ('<10m', 'evergreen_needleleaf'), ('<10m', 'evergreen_broadleaf'), ('10-20m', 'evergreen_needleleaf'), (nan, 'evergreen_needleleaf'), ('<10m', 'unknown')}
height_class          forest_type  n_folds_before  mae_before  bias_before  n_folds_after  mae_after  bias_after
        <10m  deciduous_broadleaf             117    0.259769     0.084116            117   0.260259    0.084239
        <10m  evergreen_broadleaf             209    0.257623    -0.030389            209   0.259054   -0.030453
        <10m evergreen_needleleaf               0         NaN          NaN              0        NaN         NaN
        <10m                mixed               6    0.284786    -0.284786              6   0.284691   -0.284691
      10-20m  dec

In [15]:
full_v2.to_parquet("../data/processed/all_cells_current.parquet")
print(f"saved {len(full_v2)} cells, replacing the checkpoint ({len(full)} cells before)")


saved 591659 cells, replacing the checkpoint (592048 cells before)
